In [1]:
import pandas as pd

In [65]:
df_24 = pd.read_csv("/Users/luka/priv_projects/moomotion/data/collar_weather_bolus_2024.csv")
df_25 = pd.read_csv("/Users/luka/priv_projects/moomotion/data/collar_weather_bolus_2025.csv")

/var/folders/r_/cnfqbwmx0kn_4s0yzfm9fxdh0000gn/T/ipykernel_30685/186249087.py:1: DtypeWarning: Columns (8,31,42) have mixed types. Specify dtype option on import or set low_memory=False.
  df_24 = pd.read_csv("/Users/luka/priv_projects/moomotion/data/collar_weather_bolus_2024.csv")
/var/folders/r_/cnfqbwmx0kn_4s0yzfm9fxdh0000gn/T/ipykernel_30685/186249087.py:2: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df_25 = pd.read_csv("/Users/luka/priv_projects/moomotion/data/collar_weather_bolus_2025.csv")


In [67]:
df_25.columns

Index(['neckband', 'animal_id', 'datetime', 'date', 'time', 'behaviour_1',
       'Stehen/Liegen', 'behaviour_2', 'comments', 'Unix_Timestamp',
       'GNSS_Latitude', 'GNSS_Longitude', 'Odometer_km',
       'Audio_Stimulus_Count', 'Pulse_Stimulus_Count', 'Distance_To_Fence_Max',
       'Distance_To_Fence_Min', 'IMU_Tick_Count_40mG', 'IMU_Tick_Count_80mG',
       'IMU_Tick_Count_120mG', 'IMU_Tick_Count_160mG', 'IMU_Tick_Count_200mG',
       'IMU_Tick_Count_240mG', 'Nr.', 'LOM_X', 'GEB_DATR', 'GESCHL_R', 'Herde',
       'Gewicht 03_07_2024', 'BCS 03_07_2024', 'Lahmheit', 'Kommentar',
       'eShepherd ID', 'Nr..1', 'Breed', 'Geb. Dat', 'Geschl.', 'Generation',
       'Gewicht 21.02.2024', 'BCS (1-5) _21.02.2024', 'Gewicht 03.07.2024',
       'BCS (1-5)_03.07.2024', 'Calvig Date2024', 'timestamp', 'act_index',
       'temp', 'temp_normal_index', 'heat_index', 'calving_index', 'rum_index',
       'act', 'temp_dec_index', 'temp_height_index', 'temp_inc_index',
       'temp_without_drink_cy

In [68]:
COLUMNS_TO_KEEP = ['neckband', 'animal_id', 'datetime',
       'Stehen/Liegen', 'Unix_Timestamp',
       'GNSS_Latitude', 'GNSS_Longitude', 'Odometer_km',
       'IMU_Tick_Count_40mG', 'IMU_Tick_Count_80mG',
       'IMU_Tick_Count_120mG', 'IMU_Tick_Count_160mG', 'IMU_Tick_Count_200mG',
       'IMU_Tick_Count_240mG','timestamp', 'act_index',
       'temp', 'temp_normal_index', 'heat_index', 'calving_index', 'rum_index',
       'act', 'temp_dec_index', 'temp_height_index', 'temp_inc_index',
       'temp_without_drink_cycles', 'water_intake', 'AirT_C_Avg', 'RelHumid',
       'Rain_corr_mm_Tot', 'BP_mbar_Avg', 'WindDir_deg', 'WindSpd_m_s_Avg',
       'WindSpd_m_s_Max', 'WindSpd_m_s_Min', 'Tdewpt_C_Avg', 'Twetbulb_C_Avg',
       'SunHrs_Tot', 'PotSlrRad_Avg', 'GroundT_C_Avg', 'Rad_SWin_Avg',
       'Rad_SWout_Avg']

In [69]:
df_24 = df_24[COLUMNS_TO_KEEP]

In [71]:
df_24.to_csv("/Users/luka/priv_projects/moomotion/data/CWB_2024.csv", index=False)

# Merging Individual Cow Data with IMU Data

In [ ]:
# this is the data with IMU ticks and labels, but no context data
collar_df = pd.read_csv('../data/second_batch_labels.csv', delimiter=';')
collar_df.shape

(13949, 20)

In [54]:
collar_df['datetime'] = pd.to_datetime(collar_df['datetime'], utc=True)
collar_df = collar_df[collar_df['datetime'].dt.year == 2025]

In [ ]:
if 'Stehen/Liegen' not in collar_df.columns:
    collar_df['Stehen/Liegen'] = collar_df.apply(lambda row: 'Liegen' if row['behaviour_1'] == 'Liegen' else 'Stehen', axis=1)

# Merging Weather with IMU & Individual Cow Data

In [56]:
weather_data_path = "../data/merged_smaxtec_weather_20250812_bis_20250814.csv"  # Path to the weather data
weather_df = pd.read_csv(weather_data_path)

In [ ]:
# Ensure datetime columns are timezone-naive and in datetime format
collar_df['datetime'] = (
    pd.to_datetime(collar_df['datetime'], format='mixed', utc=True)
    .dt.tz_localize(None)
)

weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'], format='mixed', utc=True).dt.tz_localize(None)

# Sort both DataFrames by their datetime columns
collar_df = collar_df.sort_values('datetime')
weather_df = weather_df.sort_values('timestamp')
# Merge using merge_asof to forward-fill weather data for up to 14 minutes (i.e., 14*60=840 seconds)
collar_weather_df = pd.merge_asof(
    collar_df,
    weather_df,
    left_on='datetime',
    right_on='timestamp',
    direction='backward',
    tolerance=pd.Timedelta('14min')
)

# Now, collar_weather_df contains weather data forward-filled for up to 14 minutes
collar_weather_df.head()

,neckband,animal_id_x,datetime,behaviour_1,behaviour_2,comments,Unix_Timestamp,GNSS_Latitude,GNSS_Longitude,Odometer_km,...,WindSpd_m_s_Max,WindSpd_m_s_Min,Tdewpt_C_Avg,Twetbulb_C_Avg,SunHrs_Tot,PotSlrRad_Avg,GroundT_C_Avg,Rad_SWin_Avg,Rad_SWout_Avg,THI
0,596764580,74863,2025-08-12 07:21:00,Laufen,Fressen,NaN,2025-08-12 07:21:00+00:00,"52,39529433333333","14,288241166666667","1,145",...,1.87,0.21,11.59,15.25,0.25,8.72,16.86,421.9,58.94,66.502725
1,1720387579,22097,2025-08-12 07:21:00,Laufen,Fressen,NaN,2025-08-12 07:21:00+00:00,"52,39584233333333","14,287210166666666","1,125",...,1.87,0.21,11.59,15.25,0.25,8.72,16.86,421.9,58.94,66.502725
2,1730919888,21809,2025-08-12 07:21:00,Laufen,Fressen,NaN,2025-08-12 07:21:00+00:00,"52,39582933333333","14,28744","1,417",...,1.87,0.21,11.59,15.25,0.25,8.72,16.86,421.9,58.94,66.502725
3,1747548909,75433,2025-08-12 07:21:00,Laufen,Fressen,NaN,2025-08-12 07:21:00+00:00,"52,395512833333335","14,288095166666666","1,193",...,1.87,0.21,11.59,15.25,0.25,8.72,16.86,421.9,58.94,66.502725
4,894016198,37464,2025-08-12 07:21:00,Laufen,Fressen,NaN,2025-08-12 07:21:00+00:00,"52,39571483333334","14,287997","1,298",...,1.87,0.21,11.59,15.25,0.25,8.72,16.86,421.9,58.94,66.502725


In [ ]:
# drop all *_y columns
collar_weather_df = collar_weather_df.loc[:, ~collar_weather_df.columns.str.endswith('_y')]

# remove trailing _x from remaining column names
collar_weather_df = collar_weather_df.rename(columns=lambda c: c[:-2] if c.endswith('_x') else c)

# sort rows so that all animal records are in chronological order
collar_weather_df.sort_values(['animal_id', 'datetime'], inplace=True)

In [ ]:
collar_weather_df.to_csv("/Users/luka/priv_projects/moomotion/data/CW_FULL_COHORT_2025.csv", index=False)

# Merging Bolus Data with IMU, Individual Cow, and Weather Data

In [43]:
bolus_data_path = "../data/smaxtec.csv"  # Path to the bolus data

bolus_df = pd.read_csv(bolus_data_path, delimiter=';', decimal=',')

In [ ]:
bolus_df['timestamp'] = pd.to_datetime(bolus_df['timestamp']).dt.tz_localize(None)
collar_weather_df = collar_weather_df.sort_values('datetime')
bolus_df = bolus_df.sort_values('timestamp')

In [ ]:
collar_weather_bolus_df = pd.merge_asof(
    collar_weather_df,
    bolus_df,
    left_on="datetime",
    right_on="timestamp",
    by="animal_id",
    direction="backward",
    tolerance=pd.Timedelta("9min"),
)

In [ ]:
collar_weather_bolus_df.sort_values(['animal_id', 'datetime'], inplace=True)

In [ ]:
animals_with_bolus = list(set(bolus_df.animal_id))

collar_weather_bolus_df = collar_weather_bolus_df[collar_weather_bolus_df['animal_id'].isin(animals_with_bolus)]

In [ ]:
collar_weather_bolus_df.shape

(17561, 180)

In [60]:
collar_herde_weather_bolus_df.to_csv("../data/FINAL_merged_collar_herde_weather_bolus.csv", index=False)